# AI4Mars U-Net inference demo

Run this notebook in Google Colab to inspect a held-out validation success case and a difficult failure case. It loads the selected checkpoint and two validation examples stored with the project; it does not load or score the official test set. The examples are the highest- and lowest-IoU records among eight fixed validation candidates.

Select **Runtime → Change runtime type → T4 GPU** for faster inference. CPU also works.

In [ ]:
from pathlib import Path
import subprocess
REPO = Path('/content/ai4mars-segmentation')
if not (REPO / 'train.py').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/CatLover9463/ai4mars-segmentation.git', str(REPO)], check=True)
%cd /content/ai4mars-segmentation
!pip -q install datasets pillow scikit-learn

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from train import CLASS_NAMES, NUM_CLASSES, UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_path = Path('models/ai4mars_best.pth')
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
config = checkpoint.get('config', {})
image_size = config.get('image_size', [192, 192])
if isinstance(image_size, int): image_size = [image_size, image_size]
model = UNet(num_classes=NUM_CLASSES, base_channels=config.get('base_channels', 32)).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Device: {device} | best epoch: {checkpoint.get("epoch")} | validation mIoU: {checkpoint.get("best_val_miou", float("nan")):.3f}')

In [ ]:
PALETTE = np.array([[156,112,73], [126,132,137], [220,183,110], [200,70,55]], dtype=np.uint8)
print('Overlay legend:', ' | '.join(f'{name}={tuple(color)}' for name, color in zip(CLASS_NAMES, PALETTE)))
def overlay(gray, labels):
    colors = PALETTE[np.clip(labels, 0, 3)]
    base = np.repeat(gray[..., None], 3, axis=2).astype(np.float32)
    result = base.copy()
    valid = labels != 255
    result[valid] = 0.52 * base[valid] + 0.48 * colors[valid]
    return result.clip(0,255).astype(np.uint8)

def inspect_example(name, caption):
    image_path = Path('demo_examples') / f'{name}_input.png'
    mask_path = Path('demo_examples') / f'{name}_mask.png'
    with Image.open(image_path) as im:
        original = im.convert('L')
        resized = original.resize((int(image_size[1]), int(image_size[0])), Image.Resampling.BILINEAR)
        x = torch.from_numpy(np.asarray(resized, dtype=np.float32).copy()/255)[None,None].to(device)
    with torch.inference_mode():
        pred = model(x).argmax(1)[0].cpu().numpy().astype(np.uint8)
    pred = np.asarray(Image.fromarray(pred).resize(original.size, Image.Resampling.NEAREST), dtype=np.uint8)
    with Image.open(mask_path) as im:
        truth = np.asarray(im.convert('L'), dtype=np.uint8)
    valid = truth != 255
    ious = []
    for class_id in range(NUM_CLASSES):
        t, p = (truth == class_id) & valid, (pred == class_id) & valid
        union = np.logical_or(t,p).sum()
        if union: ious.append(np.logical_and(t,p).sum()/union)
    print(f'{caption} | validation example IoU across present classes: {np.mean(ious):.3f}')
    fig, ax = plt.subplots(1,3,figsize=(12,4))
    ax[0].imshow(original,cmap='gray'); ax[0].set_title('Input rover image')
    ax[1].imshow(overlay(np.asarray(original),truth)); ax[1].set_title('Reference labels')
    ax[2].imshow(overlay(np.asarray(original),pred)); ax[2].set_title('U-Net prediction')
    for a in ax: a.axis('off')
    fig.suptitle(caption + ' (validation data)'); plt.tight_layout(); plt.show()

inspect_example('success', 'Best sampled validation case')
inspect_example('failure', 'Hardest sampled validation case')

## Interpretation

Compare the prediction with the reference overlay, especially for small rocks and uncertain boundaries. The two examples are validation samples selected for illustration; they are not evidence that every scene will perform similarly.